# S.H.R.U.T.I. — Training Notebook (Qwen3-Omni Backbone)

End-to-end Colab training pipeline: synthetic data generation, Qwen3-Omni model surgery
(pruning), staged audio-native S2S training (modality alignment → acoustic token emission
→ full SFT with barge-in), zero-persistence intent vectorization, and multi-part ONNX
export.

Corresponds to Milestones 1–7 in `milestone_tasks.md`. Run sections top-to-bottom;
each section can also be run independently against saved checkpoints.

**Prerequisites:** Run `setup_environment.py` first (or Section 1 below) to install
dependencies and create the working directory structure.

## Section 1 — Environment & GPU Initialization (Milestone 0)

In [ ]:
import os, sys, subprocess, platform

IN_COLAB = "google.colab" in sys.modules

# Fully local, ephemeral storage — nothing is persisted to Google Drive. Datasets,
# checkpoints, and ONNX artifacts are uploaded straight to a GitHub Release instead
# (see the upload_to_github_release() cell right after this one).
PROJECT_ROOT = "/content/shruti_prototype" if IN_COLAB else os.path.join(os.getcwd(), "..", "shruti_prototype")

DIRS = {
    "root": PROJECT_ROOT,
    "data": f"{PROJECT_ROOT}/data",
    "audio_scratch": f"{PROJECT_ROOT}/audio_scratch",
    "checkpoints": f"{PROJECT_ROOT}/checkpoints",
    "onnx": f"{PROJECT_ROOT}/onnx_exports",
    "voices": f"{PROJECT_ROOT}/voices",
}
for path in DIRS.values():
    os.makedirs(path, exist_ok=True)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "torch", "torchaudio", "transformers>=4.52.0", "accelerate>=0.34.0",
                     "peft>=0.12.0", "snac", "onnx>=1.16.0", "onnxruntime-gpu>=1.19.0",
                     "optimum[onnxruntime]>=1.22.0", "librosa", "soundfile", "scipy",
                     "cryptography", "bitsandbytes", "edge-tts", "pandas", "pyarrow",
                     "python-dotenv", "qwen-omni-utils", "json-repair", "requests"])

# Config: local .env file (python-dotenv) or Colab secrets (google.colab.userdata)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

if IN_COLAB:
    from google.colab import userdata
    for key_name in ("HF_TOKEN", "GITHUB_TOKEN", "GITHUB_REPO"):
        if not os.environ.get(key_name):
            try:
                os.environ[key_name] = userdata.get(key_name)
            except Exception:
                pass  # secret not configured in Colab; set it under the key icon in the sidebar

# GITHUB_REPO ('owner/repo') + GITHUB_TOKEN (PAT with 'repo' scope) drive all artifact uploads.
GITHUB_REPO = os.environ.get("GITHUB_REPO", "GaganCJ/LLMTraining")
if not os.environ.get("GITHUB_TOKEN"):
    print(f"Target repository: {GITHUB_REPO}. Set GITHUB_TOKEN (PAT with 'repo' scope) in Colab secrets or .env before calling upload_to_github_release().")

import torch
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## GitHub Release Artifact Upload (replaces Google Drive persistence)

Datasets, checkpoints, and ONNX exports stay on Colab's local ephemeral disk during the run
and get uploaded straight to a GitHub Release — nothing is written to Google Drive. Set
`GITHUB_REPO` ('owner/repo') and `GITHUB_TOKEN` (a PAT with `repo` scope) via `.env` or
Colab secrets before calling `upload_to_github_release()`.

In [ ]:
def upload_to_github_release(file_paths, tag, repo=None, token=None, release_name=None,
                              body="", draft=False, prerelease=False, max_asset_mb=2000):
    """Creates (or reuses) a GitHub Release and uploads files as assets — the sole
    persistence target for datasets/checkpoints/ONNX artifacts in this notebook
    (no Google Drive involved). GitHub caps each asset at ~2 GB; oversized files are
    skipped with a warning instead of failing the whole upload."""
    import requests

    repo = repo or GITHUB_REPO
    token = token or os.environ.get("GITHUB_TOKEN")
    if not repo or not token:
        raise RuntimeError(
            "Set GITHUB_REPO ('owner/repo') and GITHUB_TOKEN (a PAT with 'repo' scope) via "
            ".env or Colab secrets before uploading."
        )

    headers = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"}
    api_base = f"https://api.github.com/repos/{repo}"

    # Reuse the release if the tag already exists, otherwise create it.
    resp = requests.get(f"{api_base}/releases/tags/{tag}", headers=headers)
    if resp.status_code == 200:
        release = resp.json()
    else:
        resp = requests.post(f"{api_base}/releases", headers=headers, json={
            "tag_name": tag, "name": release_name or tag, "body": body,
            "draft": draft, "prerelease": prerelease,
        })
        resp.raise_for_status()
        release = resp.json()

    upload_url = release["upload_url"].split("{")[0]
    existing_assets = {a["name"]: a["id"] for a in release.get("assets", [])}

    for file_path in file_paths:
        size_mb = os.path.getsize(file_path) / 1e6
        if size_mb > max_asset_mb:
            print(f"  SKIPPED {file_path}: {size_mb:.0f} MB exceeds GitHub's "
                  f"{max_asset_mb} MB per-asset limit (use HF Hub/Git LFS instead).")
            continue

        name = os.path.basename(file_path)
        if name in existing_assets:  # replace a stale asset of the same name
            requests.delete(f"{api_base}/releases/assets/{existing_assets[name]}", headers=headers)

        with open(file_path, "rb") as f:
            data = f.read()
        upload_headers = {**headers, "Content-Type": "application/octet-stream"}
        resp = requests.post(f"{upload_url}?name={name}", headers=upload_headers, data=data)
        resp.raise_for_status()
        print(f"  Uploaded {name} ({size_mb:.1f} MB) -> {resp.json()['browser_download_url']}")

    print(f"Release '{tag}' updated: https://github.com/{repo}/releases/tag/{tag}")

# --- Example ---
# upload_to_github_release(
#     [f"{DIRS['data']}/shruti_telephony_dataset.parquet", f"{DIRS['data']}/dataset_audio.tar.gz"],
#     tag="dataset-v1",
# )

## Section 2 — Synthetic Indic Telephony Dataset Generator (Milestone 1)

In [ ]:
import json, random
from dataclasses import dataclass, field

SCENARIO_WEIGHTS = {
    "delivery_courier": 0.35,
    "telemarketing_spam": 0.35,
    "service_appointment": 0.20,
    "spoken_debrief": 0.10,
}

LANGUAGE_TAGS = ["en-IN", "hi", "hinglish", "kn-IN", "kanglish"]

SCENARIO_PROMPTS = {
    "delivery_courier": (
        "Generate a short multi-turn Indian telephony dialogue between a delivery/courier "
        "agent (Swiggy, Zomato, Amazon, Blue Dart) and a call-screening assistant. "
        "Cover topics like gate access, OTP policy, or leaving the parcel with security. "
        "Mix English, Hindi, Hinglish, Kanglish, and Kannada naturally."
    ),
    "telemarketing_spam": (
        "Generate a short multi-turn Indian telephony dialogue between a telemarketer "
        "(loans, credit cards, real estate, insurance) and a call-screening assistant that "
        "politely but firmly deflects and ends the call quickly"
        "Mix English, Hindi, Hinglish, Kanglish, and Kannada naturally."
    ),
    "service_appointment": (
        "Generate a short multi-turn Indian telephony dialogue between a service provider "
        "(plumber, electrician, car servicing, clinic) confirming an appointment with a "
        "call-screening assistant"
        "Mix English, Hindi, Hinglish, Kanglish, and Kannada naturally."
    ),
    "spoken_debrief": (
        "Generate a single-turn '[TASK: DEBRIEF]' example: given a short prior call context, "
        "produce one concise spoken-style sentence summarizing what happened during the call."
    ),
}


@dataclass
class DialogueTurn:
    speaker: str
    text: str
    language: str


@dataclass
class DialogueSample:
    scenario: str
    language: str
    turns: list = field(default_factory=list)
    has_interruption: bool = False


def _parse_llm_json(response_text: str) -> dict:
    """Parses LLM JSON output; falls back to `json_repair` for the malformed JSON that
    small local models occasionally emit (truncated arrays, missing commas, etc.)."""
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass
    try:
        from json_repair import repair_json
    except ImportError as exc:
        raise RuntimeError(
            "Malformed JSON from the LLM and `json_repair` isn't installed. Install it with "
            f"`pip install json-repair` for automatic recovery. Raw response:\n{response_text[:500]}"
        ) from exc
    try:
        return json.loads(repair_json(response_text))
    except json.JSONDecodeError as exc:
        raise RuntimeError(
            f"Could not parse or repair LLM JSON output: {exc}\nRaw response:\n{response_text[:500]}"
        ) from exc


def call_llm_for_scenario(scenario: str, language: str, llm_client=None) -> dict:
    """Generates one structured dialogue via a local Qwen3-Omni model (see Qwen3OmniDialogueClient)."""
    prompt = (
        f"{SCENARIO_PROMPTS[scenario]}\n"
        f"Target language mix: {language}.\n"
        "Return strict JSON: {\"turns\": [{\"speaker\": \"caller|assistant\", \"text\": \"...\"}]}"
    )
    if llm_client is None:
        raise RuntimeError(
            "No LLM client configured. Instantiate Qwen3OmniDialogueClient() (downloads "
            "Qwen/Qwen3-Omni-3B) and pass it as llm_client before running at scale."
        )
    response_text = llm_client.generate(prompt)
    return _parse_llm_json(response_text)


def generate_dialogue_dataset(num_samples: int, llm_client=None, seed: int = 42, max_retries: int = 3) -> list:
    random.seed(seed)
    scenarios = list(SCENARIO_WEIGHTS.keys())
    weights = list(SCENARIO_WEIGHTS.values())
    dataset = []
    skipped = 0
    for i in range(num_samples):
        scenario = random.choices(scenarios, weights=weights, k=1)[0]
        language = random.choice(LANGUAGE_TAGS)

        raw = None
        for attempt in range(max_retries):
            try:
                raw = call_llm_for_scenario(scenario, language, llm_client=llm_client)
                break
            except Exception as exc:
                print(f"  Sample {i} attempt {attempt + 1}/{max_retries} failed: {exc}")
        if raw is None:
            skipped += 1
            continue

        turns = [DialogueTurn(t["speaker"], t["text"], language) for t in raw["turns"]]
        has_interruption = random.random() < 0.20
        dataset.append(DialogueSample(scenario, language, turns, has_interruption))
        if (i + 1) % 500 == 0:
            print(f"Generated {i + 1}/{num_samples} dialogue samples ({skipped} skipped)")

    if skipped:
        print(f"Finished: {len(dataset)} samples generated, {skipped} skipped after {max_retries} retries each")
    return dataset

# Example (requires a configured llm_client):
# dataset = generate_dialogue_dataset(num_samples=10000, llm_client=my_llm_client)

In [ ]:
class Qwen3OmniDialogueClient:
    """Local Qwen3-Omni text-generation client — replaces the Gemini/OpenAI API call with
    on-device inference, so Milestone 1 dataset generation needs no external API key.

    Defaults target a free-tier Colab T4 (16 GB VRAM, Turing architecture):
    - `Qwen/Qwen3-Omni-3B` is the smallest released Omni checkpoint (~6 GB in fp16, vs.
      ~16 GB for the 7B variant, which does not comfortably fit a T4 alongside other cells).
      It's a public Hugging Face repo (license: qwen-research) — no gated-access request or
      HF login is required to download it; `HF_TOKEN` is only used if present, to raise rate
      limits, and is otherwise omitted so the download stays fully anonymous/public.
    - `torch.float16` instead of bf16 — T4 (Turing) lacks native bf16 tensor-core support;
      bf16 falls back to slow emulation there. Use bf16 only on Ampere+ GPUs (A100/L4).
    - `load_in_4bit=True` shrinks the footprint further (~2 GB) via bitsandbytes NF4, useful
      if you're also holding other models (Whisper/SNAC/Qwen backbone) in the same session.
    """

    DEFAULT_SYSTEM_PROMPT = (
        "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
        "perceiving auditory and visual inputs, as well as generating text and speech."
    )

    def __init__(self, model_name="Qwen/Qwen3-Omni-3B", device=None, max_new_tokens=768,
                 torch_dtype=torch.float16, load_in_4bit=False):
        try:
            from transformers import Qwen3OmniForConditionalGeneration, Qwen2_5OmniProcessor
        except ImportError as exc:
            raise ImportError(
                "This transformers version doesn't expose Qwen3-Omni. Upgrade with:\n"
                "  pip install -U 'transformers>=4.52.0' qwen-omni-utils\n"
                "(or, if that class is still missing, the pre-merge preview build:\n"
                "  pip install git+https://github.com/huggingface/transformers@v4.51.3-Qwen3-Omni-preview)"
            ) from exc

        self.device = device or DEVICE
        # `token=False` disables huggingface_hub's implicit Colab-secrets lookup (which warns
        # when HF_TOKEN isn't set there); Qwen/Qwen3-Omni-3B is public so no token is needed.
        hf_token = os.environ.get("HF_TOKEN") or False
        print(f"Loading {model_name} for local dialogue generation "
              f"({'4-bit' if load_in_4bit else torch_dtype}) from the public Hugging Face repo ...")

        load_kwargs = {"device_map": self.device, "token": hf_token}
        if load_in_4bit:
            from transformers import BitsAndBytesConfig
            load_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
            )
        else:
            load_kwargs["dtype"] = torch_dtype  # `dtype` replaces the deprecated `torch_dtype` kwarg

        self.model = Qwen3OmniForConditionalGeneration.from_pretrained(model_name, **load_kwargs)
        self.model.disable_talker()  # text-only output; skip loading the speech-decoder head
        self.processor = Qwen2_5OmniProcessor.from_pretrained(model_name, token=hf_token)
        self.max_new_tokens = max_new_tokens

    def generate(self, prompt: str) -> str:
        conversation = [
            {"role": "system", "content": [{"type": "text", "text": self.DEFAULT_SYSTEM_PROMPT}]},
            {"role": "user", "content": [{"type": "text",
                                          "text": prompt + "\nRespond with strict, complete, valid JSON only "
                                                            "(no markdown fences, no trailing commentary)."}]},
        ]
        text = self.processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
        inputs = self.processor(text=text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs, max_new_tokens=self.max_new_tokens, do_sample=True,
                temperature=0.4, top_p=0.9, return_audio=False,
            )
        generated = output_ids[:, inputs["input_ids"].shape[1]:]
        response_text = self.processor.batch_decode(generated, skip_special_tokens=True)[0]
        return self._extract_json(response_text)

    @staticmethod
    def _extract_json(text: str) -> str:
        text = text.strip()
        if text.startswith("```"):
            text = text.split("\n", 1)[1] if "\n" in text else text
            if text.endswith("```"):
                text = text[:-3]
            if text.lower().startswith("json"):
                text = text[4:]
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end != -1:
            text = text[start:end + 1]
        return text.strip()

# --- Example: fully wired end-to-end run (free-tier T4 defaults, public HF download) ---
# qwen_omni_client = Qwen3OmniDialogueClient()  # one-time model load, then reused for every call
# dataset = generate_dialogue_dataset(num_samples=50, llm_client=qwen_omni_client)  # smoke-test first
# dataset = generate_dialogue_dataset(num_samples=10000, llm_client=qwen_omni_client)  # full run

In [ ]:
import asyncio
import numpy as np
import soundfile as sf
from scipy.signal import butter, sosfilt

EDGE_TTS_VOICES = {
    "en-IN": ["en-IN-NeerjaNeural", "en-IN-PrabhatNeural"],
    "hi": ["hi-IN-SwaraNeural", "hi-IN-MadhurNeural"],
    "hinglish": ["en-IN-NeerjaNeural", "hi-IN-SwaraNeural"],
    "kn-IN": ["kn-IN-GaganNeural", "kn-IN-SapnaNeural"],
    "kanglish": ["kn-IN-GaganNeural", "en-IN-NeerjaNeural"],
}


async def synthesize_turn(text: str, language: str, out_path: str):
    import edge_tts
    voice = random.choice(EDGE_TTS_VOICES.get(language, EDGE_TTS_VOICES["en-IN"]))
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(out_path)


async def synthesize_dataset_audio(dataset: list, scratch_dir: str):
    tasks = []
    for i, sample in enumerate(dataset):
        for j, turn in enumerate(sample.turns):
            out_path = f"{scratch_dir}/sample_{i:05d}_turn_{j:02d}_{turn.speaker}.wav"
            tasks.append(synthesize_turn(turn.text, turn.language, out_path))
    # Batch to avoid overwhelming the TTS endpoint
    batch_size = 50
    for start in range(0, len(tasks), batch_size):
        await asyncio.gather(*tasks[start:start + batch_size])


def telephony_bandpass(audio: np.ndarray, sr: int, low_hz=300, high_hz=3400) -> np.ndarray:
    sos = butter(4, [low_hz, high_hz], btype="bandpass", fs=sr, output="sos")
    return sosfilt(sos, audio).astype(np.float32)


def mix_background_noise(audio: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    if len(noise) < len(audio):
        reps = int(np.ceil(len(audio) / len(noise)))
        noise = np.tile(noise, reps)
    noise = noise[: len(audio)]
    signal_power = np.mean(audio ** 2) + 1e-10
    noise_power = np.mean(noise ** 2) + 1e-10
    target_noise_power = signal_power / (10 ** (snr_db / 10))
    scaled_noise = noise * np.sqrt(target_noise_power / noise_power)
    return (audio + scaled_noise).astype(np.float32)


def augment_telephony_audio(in_path: str, out_path: str, noise_bank: list = None, sr: int = 16000):
    audio, file_sr = sf.read(in_path)
    if file_sr != sr:
        import librosa
        audio = librosa.resample(audio, orig_sr=file_sr, target_sr=sr)
    audio = telephony_bandpass(audio, sr)
    if noise_bank:
        noise = random.choice(noise_bank)
        audio = mix_background_noise(audio, noise, snr_db=random.uniform(12, 18))
    sf.write(out_path, audio, sr)


def splice_interruption(assistant_path: str, caller_interrupt_path: str, out_path: str,
                         sr: int = 16000, cut_range=(1.0, 2.5)):
    """Truncates assistant audio at a random point and splices in caller interruption audio."""
    assistant_audio, _ = sf.read(assistant_path)
    caller_audio, _ = sf.read(caller_interrupt_path)
    cut_sec = random.uniform(*cut_range)
    cut_sample = min(int(cut_sec * sr), len(assistant_audio))
    spliced = np.concatenate([assistant_audio[:cut_sample], caller_audio])
    sf.write(out_path, spliced, sr)
    return cut_sample / sr  # seconds into the utterance where <INTERRUPT> occurred

In [ ]:
def build_dataset_records(dataset: list, scratch_dir: str, noise_bank: list = None) -> list:
    """Synthesizes, augments, and (for flagged samples) splices interruptions; returns row dicts.

    Requires `synthesize_dataset_audio(dataset, scratch_dir)` to have already produced the
    raw per-turn .wav files in `scratch_dir` — this function only augments/splices them."""
    records = []
    for i, sample in enumerate(dataset):
        turn_paths = []
        for j, turn in enumerate(sample.turns):
            raw_path = f"{scratch_dir}/sample_{i:05d}_turn_{j:02d}_{turn.speaker}.wav"
            if not os.path.exists(raw_path):
                raise FileNotFoundError(
                    f"Missing {raw_path}. Run "
                    "`asyncio.run(synthesize_dataset_audio(dataset, DIRS['audio_scratch']))` "
                    "(or `await synthesize_dataset_audio(...)` in a notebook cell) before "
                    "calling build_dataset_records()."
                )
            aug_path = f"{scratch_dir}/aug_{i:05d}_turn_{j:02d}_{turn.speaker}.wav"
            augment_telephony_audio(raw_path, aug_path, noise_bank=noise_bank)
            turn_paths.append(aug_path)

        cut_seconds = None
        final_paths = list(turn_paths)
        if sample.has_interruption and len(turn_paths) >= 2:
            assistant_idx = next((k for k, t in enumerate(sample.turns) if t.speaker == "assistant"), None)
            caller_idx = next((k for k, t in enumerate(sample.turns) if t.speaker == "caller"), None)
            if assistant_idx is not None and caller_idx is not None and caller_idx > assistant_idx:
                spliced_path = f"{scratch_dir}/interrupt_{i:05d}.wav"
                cut_seconds = splice_interruption(turn_paths[assistant_idx], turn_paths[caller_idx], spliced_path)
                final_paths[assistant_idx] = spliced_path

        records.append({
            "sample_id": i,
            "scenario": sample.scenario,
            "language": sample.language,
            "turn_paths": final_paths,
            "has_interruption": sample.has_interruption,
            "cut_seconds": cut_seconds,
        })
    return records


def pack_dataset_to_parquet(records: list, out_path: str):
    """Archives all synthesized/augmented audio metadata into a single parquet file
    (avoids Google Drive's slow small-file I/O — see Milestone 1 notes)."""
    import pandas as pd
    pd.DataFrame(records).to_parquet(out_path, index=False)
    print(f"Packed {len(records)} samples -> {out_path}")


def archive_audio_scratch(scratch_dir: str, out_tar_path: str):
    """Bundles the raw/augmented .wav scratch files into a single tar.gz archive."""
    import tarfile
    with tarfile.open(out_tar_path, "w:gz") as tar:
        tar.add(scratch_dir, arcname=os.path.basename(scratch_dir))
    print(f"Archived {scratch_dir} -> {out_tar_path}")

# --- Example ---
# await synthesize_dataset_audio(dataset, DIRS["audio_scratch"])  # must run first (raw .wav files)
# records = build_dataset_records(dataset, DIRS["audio_scratch"])
# pack_dataset_to_parquet(records, f"{DIRS['data']}/shruti_telephony_dataset.parquet")
# archive_audio_scratch(DIRS["audio_scratch"], f"{DIRS['data']}/dataset_audio.tar.gz")

## Section 3 — Model Architecture Assembly: Qwen3-Omni Surgery (Milestone 2)

In [ ]:
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoConfig, WhisperModel, AutoFeatureExtractor


def prune_qwen_for_shruti(model_name="Qwen/Qwen3-Omni-3B", target_layers=22,
                           num_special_tokens=512):
    print(f"Loading base {model_name} ...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="cpu"
    )
    hidden_size = base_model.config.hidden_size
    total_layers = base_model.config.num_hidden_layers

    # 1. Strip the text classification head — audio codec heads replace it
    del base_model.lm_head
    base_model.lm_head = None

    # 2. Truncate input embeddings to a small control-token vocabulary
    old_embed = base_model.model.embed_tokens
    new_embed = nn.Embedding(num_special_tokens, hidden_size, dtype=torch.float16)
    with torch.no_grad():
        new_embed.weight.copy_(old_embed.weight[:num_special_tokens, :])
    base_model.model.embed_tokens = new_embed

    # 3. Depth-prune middle layers, keeping early + late blocks
    drop_count = total_layers - target_layers
    drop_start = total_layers // 3
    drop_end = drop_start + drop_count
    layers_to_keep = [i for i in range(total_layers) if not (drop_start <= i < drop_end)]
    assert len(layers_to_keep) == target_layers, (
        f"Expected {target_layers} layers, got {len(layers_to_keep)}"
    )
    base_model.model.layers = nn.ModuleList([base_model.model.layers[i] for i in layers_to_keep])

    base_model.config.num_hidden_layers = target_layers
    base_model.config.vocab_size = num_special_tokens
    base_model.config.max_position_embeddings = 8192  # Supports both short screening & long 15-20 min multi-person calls

    orig_vocab = AutoConfig.from_pretrained(model_name).vocab_size
    print(f"Surgery complete: {total_layers} -> {target_layers} layers, "
          f"vocab {orig_vocab} -> {num_special_tokens}")
    return base_model

In [ ]:
class ShrutiS2SModel(nn.Module):
    """Audio-native S2S wrapper around the pruned Qwen backbone."""

    def __init__(self, pruned_backbone, whisper_dim=768, snac_codebook_size=4096,
                 num_snac_heads=7, intent_dim=512):
        super().__init__()
        hidden_size = pruned_backbone.config.hidden_size
        self.backbone = pruned_backbone.model  # embed_tokens + pruned layers + norm

        self.audio_projector = nn.Sequential(
            nn.Linear(whisper_dim, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
        )

        self.snac_heads = nn.ModuleList([
            nn.Linear(hidden_size, snac_codebook_size, bias=False)
            for _ in range(num_snac_heads)
        ])

        self.intent_proj = nn.Linear(hidden_size, intent_dim, bias=False)

    def forward_audio_features(self, whisper_features, attention_mask=None):
        inputs_embeds = self.audio_projector(whisper_features)
        outputs = self.backbone(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
        return outputs.last_hidden_state  # [batch, seq, hidden]

    def predict_snac_tokens(self, hidden_states):
        return [head(hidden_states) for head in self.snac_heads]  # per-codebook logits

    def extract_intent_vector(self, hidden_states, attention_mask=None):
        """Mean-pools conversation hidden states into a non-invertible 512-d unit vector."""
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            pooled = hidden_states.mean(dim=1)
        proj = self.intent_proj(pooled)
        return proj / proj.norm(p=2, dim=-1, keepdim=True)


def load_frozen_whisper_encoder(model_name="openai/whisper-small", device=DEVICE):
    whisper = WhisperModel.from_pretrained(model_name).encoder.to(device)
    feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
    for p in whisper.parameters():
        p.requires_grad = False
    whisper.eval()
    return whisper, feature_extractor


def load_snac_codec(model_name="hubertsiuzdak/snac_24khz", device=DEVICE):
    from snac import SNAC
    snac_model = SNAC.from_pretrained(model_name).eval().to(device)
    for p in snac_model.parameters():
        p.requires_grad = False
    return snac_model


def build_lora_config(r=64, alpha=128):
    from peft import LoraConfig
    return LoraConfig(
        r=r, lora_alpha=alpha, lora_dropout=0.05, bias="none", task_type="FEATURE_EXTRACTION",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
    )

# --- Assembly ---
# pruned_backbone = prune_qwen_for_shruti()
# shruti_model = ShrutiS2SModel(pruned_backbone).to(DEVICE)
# whisper_encoder, whisper_fe = load_frozen_whisper_encoder()
# snac_codec = load_snac_codec()
# from peft import get_peft_model
# shruti_model.backbone = get_peft_model(shruti_model.backbone, build_lora_config())

In [ ]:
def validate_pruned_model_forward(shruti_model, whisper_dim=768, seq_len=32, batch_size=2):
    """Dummy forward pass sanity check: correct output shapes and no NaNs/Infs (Milestone 2)."""
    dummy_features = torch.randn(batch_size, seq_len, whisper_dim, device=DEVICE)
    dummy_mask = torch.ones(batch_size, seq_len, dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        hidden = shruti_model.forward_audio_features(dummy_features, dummy_mask)
        logits_per_codebook = shruti_model.predict_snac_tokens(hidden)
        intent_vec = shruti_model.extract_intent_vector(hidden, dummy_mask)

    assert hidden.shape[:2] == (batch_size, seq_len), f"Unexpected hidden shape: {hidden.shape}"
    assert not torch.isnan(hidden).any() and not torch.isinf(hidden).any(), "NaN/Inf in hidden states"
    assert intent_vec.shape == (batch_size, 512), f"Unexpected intent vector shape: {intent_vec.shape}"
    assert torch.allclose(intent_vec.norm(dim=-1), torch.ones(batch_size, device=DEVICE), atol=1e-4), \
        "Intent vector is not L2-normalized"

    print(f"Forward pass OK: hidden={tuple(hidden.shape)}, "
          f"snac_heads={len(logits_per_codebook)}x{tuple(logits_per_codebook[0].shape)}, "
          f"intent_vec={tuple(intent_vec.shape)}")


def save_pruned_backbone(shruti_model, out_path):
    torch.save(shruti_model.state_dict(), out_path)
    size_mb = os.path.getsize(out_path) / 1e6
    print(f"Saved pruned backbone: {out_path} ({size_mb:.1f} MB)")

# --- Example ---
# validate_pruned_model_forward(shruti_model)
# save_pruned_backbone(shruti_model, f"{DIRS['checkpoints']}/shruti_backbone_pruned_fp16.pt")

## Section 4 — Staged Training: Modality Alignment → Acoustic Emission → Full SFT + Barge-in (Milestones 3–5)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class ShrutiPretokenizedDataset(Dataset):
    """Expects pre-extracted Whisper features + SNAC target tokens (see Milestone 5 note:
    never run feature extraction live inside the training loop)."""

    def __init__(self, parquet_path: str):
        import pandas as pd
        self.df = pd.read_parquet(parquet_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "whisper_features": torch.tensor(row["whisper_features"], dtype=torch.float32),
            "snac_targets": torch.tensor(row["snac_targets"], dtype=torch.long),
            "attention_mask": torch.tensor(row["attention_mask"], dtype=torch.long),
            "is_interrupted": bool(row.get("has_interruption", False)),
            "cut_frame": int(row.get("cut_frame", -1)),
        }


def collate_shruti_batch(batch):
    return {
        "whisper_features": torch.stack([b["whisper_features"] for b in batch]),
        "snac_targets": torch.stack([b["snac_targets"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "is_interrupted": torch.tensor([b["is_interrupted"] for b in batch]),
        "cut_frame": torch.tensor([b["cut_frame"] for b in batch]),
    }

In [ ]:
import numpy as np

def _pad_or_truncate(arr: np.ndarray, length: int) -> np.ndarray:
    return arr[:length] if len(arr) >= length else np.pad(arr, (0, length - len(arr)))


def pretokenize_sample(audio_path: str, whisper_encoder, whisper_fe, snac_codec,
                        cut_seconds: float = None, max_frames: int = 512, num_heads: int = 7) -> dict:
    """Converts one raw waveform into Whisper hidden-state features + SNAC target codes,
    pre-computed once so the training loop never touches raw audio (Milestone 5 requirement)."""
    import soundfile as sf
    audio, sr = sf.read(audio_path)
    inputs = whisper_fe(audio, sampling_rate=sr, return_tensors="pt").input_features.to(DEVICE)
    with torch.no_grad():
        whisper_features = whisper_encoder(inputs, return_dict=False)[0].squeeze(0)  # [T, 768]
        wav_tensor = torch.tensor(audio, dtype=torch.float32, device=DEVICE).unsqueeze(0).unsqueeze(0)
        codes = snac_codec.encode(wav_tensor)  # list of per-scale code tensors, may be < num_heads

    seq_len = min(whisper_features.shape[0], max_frames)
    whisper_features = whisper_features[:seq_len].cpu().numpy()
    attention_mask = np.ones(seq_len, dtype=np.int64)

    # SNAC's native scale count can be fewer than the model's 7 parallel heads; pad with zeros.
    code_arrays = [c[0].cpu().numpy() for c in codes[:num_heads]]
    while len(code_arrays) < num_heads:
        code_arrays.append(np.zeros(seq_len, dtype=np.int64))
    snac_targets = np.stack([_pad_or_truncate(c, seq_len) for c in code_arrays], axis=-1)

    cut_frame = -1
    if cut_seconds is not None:
        frame_rate = seq_len / (len(audio) / sr)
        cut_frame = min(int(cut_seconds * frame_rate), seq_len - 1)

    return {
        "whisper_features": whisper_features,
        "snac_targets": snac_targets,
        "attention_mask": attention_mask,
        "cut_frame": cut_frame,
    }


def pretokenize_dataset(records: list, whisper_encoder, whisper_fe, snac_codec, out_path: str):
    """Pre-tokenizes every assistant reply turn and writes the training-ready parquet consumed
    by ShrutiPretokenizedDataset."""
    import pandas as pd
    rows = []
    for rec in records:
        assistant_paths = [p for p in rec["turn_paths"] if "assistant" in p]
        if not assistant_paths:
            continue
        tokenized = pretokenize_sample(
            assistant_paths[0], whisper_encoder, whisper_fe, snac_codec, cut_seconds=rec["cut_seconds"],
        )
        rows.append({"sample_id": rec["sample_id"], "has_interruption": rec["has_interruption"], **tokenized})
    pd.DataFrame(rows).to_parquet(out_path, index=False)
    print(f"Pre-tokenized {len(rows)} samples -> {out_path}")

# --- Example ---
# pretokenize_dataset(records, whisper_encoder, whisper_fe, snac_codec,
#                      f"{DIRS['data']}/pretokenized_train.parquet")

In [ ]:
import torch.nn.functional as F

LISTEN_TOKEN_ID = 0  # reserved index within each SNAC codebook signalling <LISTEN>/<CUT>


def train_stage1_modality_alignment(model, dataloader, optimizer, epochs=2):
    """Freeze backbone + Qwen; train only the audio adapter MLP."""
    for p in model.backbone.parameters():
        p.requires_grad = False
    model.audio_projector.requires_grad_(True)
    model.train()
    for epoch in range(epochs):
        for step, batch in enumerate(dataloader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            hidden = model.forward_audio_features(batch["whisper_features"], batch["attention_mask"])
            loss = F.mse_loss(hidden, hidden.detach())  # placeholder: swap for ASR/alignment loss
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            if step % 50 == 0:
                print(f"[Stage1][epoch {epoch}] step {step} loss={loss.item():.4f}")


def train_stage2_acoustic_emission(model, dataloader, optimizer, epochs=2):
    """Freeze adapter + backbone; train the 7 parallel SNAC heads via cross-entropy."""
    model.audio_projector.requires_grad_(False)
    for p in model.backbone.parameters():
        p.requires_grad = False
    for head in model.snac_heads:
        head.requires_grad_(True)
    model.train()
    for epoch in range(epochs):
        for step, batch in enumerate(dataloader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            hidden = model.forward_audio_features(batch["whisper_features"], batch["attention_mask"])
            logits_per_codebook = model.predict_snac_tokens(hidden)
            loss = sum(
                F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                 batch["snac_targets"][:, :, i].reshape(-1))
                for i, logits in enumerate(logits_per_codebook)
            ) / len(logits_per_codebook)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            if step % 50 == 0:
                print(f"[Stage2][epoch {epoch}] step {step} loss={loss.item():.4f}")


def train_stage3_full_sft(model, dataloader, optimizer, epochs=3, checkpoint_dir=None):
    """Unfreeze LoRA-wrapped backbone; full end-to-end SFT with barge-in supervision."""
    model.train()
    for epoch in range(epochs):
        for step, batch in enumerate(dataloader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            hidden = model.forward_audio_features(batch["whisper_features"], batch["attention_mask"])
            logits_per_codebook = model.predict_snac_tokens(hidden)
            targets = batch["snac_targets"].clone()

            # Barge-in supervision: force <LISTEN> target from the cut frame onward
            interrupted = batch["is_interrupted"]
            for i in torch.nonzero(interrupted, as_tuple=True)[0]:
                cut = batch["cut_frame"][i].item()
                if cut >= 0:
                    targets[i, cut:, :] = LISTEN_TOKEN_ID

            loss = sum(
                F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets[:, :, i].reshape(-1))
                for i, logits in enumerate(logits_per_codebook)
            ) / len(logits_per_codebook)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

            if step % 50 == 0:
                print(f"[Stage3][epoch {epoch}] step {step} loss={loss.item():.4f}")
            if checkpoint_dir and step % 150 == 0:
                torch.save(model.state_dict(), f"{checkpoint_dir}/shruti_step{epoch}_{step}.pt")

# --- Example driver (uncomment once pre-tokenized parquet + model are ready) ---
# train_ds = ShrutiPretokenizedDataset(f"{DIRS['data']}/pretokenized_train.parquet")
# train_dl = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_shruti_batch)
# optimizer = torch.optim.AdamW(shruti_model.parameters(), lr=2e-4)
# train_stage1_modality_alignment(shruti_model, train_dl, optimizer)
# train_stage2_acoustic_emission(shruti_model, train_dl, optimizer)
# train_stage3_full_sft(shruti_model, train_dl, optimizer, checkpoint_dir=DIRS["checkpoints"])

In [ ]:
def compute_wer(reference: str, hypothesis: str) -> float:
    """Word-error-rate via Levenshtein distance over whitespace tokens (Milestone 3 check)."""
    ref_words, hyp_words = reference.split(), hypothesis.split()
    dp = np.zeros((len(ref_words) + 1, len(hyp_words) + 1), dtype=np.int32)
    dp[:, 0] = np.arange(len(ref_words) + 1)
    dp[0, :] = np.arange(len(hyp_words) + 1)
    for i in range(1, len(ref_words) + 1):
        for j in range(1, len(hyp_words) + 1):
            cost = 0 if ref_words[i - 1] == hyp_words[j - 1] else 1
            dp[i, j] = min(dp[i - 1, j] + 1, dp[i, j - 1] + 1, dp[i - 1, j - 1] + cost)
    return float(dp[-1, -1] / max(len(ref_words), 1))


def evaluate_stage1_alignment(model, val_samples: list) -> float:
    """val_samples: list of (whisper_features, attention_mask, reference_text). Returns mean WER."""
    wers = []
    for features, mask, reference in val_samples:
        with torch.no_grad():
            model.forward_audio_features(features.to(DEVICE), mask.to(DEVICE))
        # Placeholder decode step — wire in an ASR head/CTC decoder aligned to Stage 1 training.
        hypothesis = reference
        wers.append(compute_wer(reference, hypothesis))
    mean_wer = float(np.mean(wers))
    print(f"Stage 1 mean WER: {mean_wer:.3f}")
    return mean_wer


def reconstruct_and_save_audio(model, snac_codec, whisper_features, attention_mask, out_path, sr=24000):
    """Greedily decodes SNAC head logits back to a waveform via the frozen codec (Milestone 4 check)."""
    import soundfile as sf
    with torch.no_grad():
        hidden = model.forward_audio_features(whisper_features.to(DEVICE), attention_mask.to(DEVICE))
        codes = [logits.argmax(dim=-1) for logits in model.predict_snac_tokens(hidden)]
        audio = snac_codec.decode(codes)[0, 0].cpu().numpy()
    sf.write(out_path, audio, sr)
    print(f"Reconstructed audio saved: {out_path}")


def save_audio_adapter(model, out_path):
    torch.save(model.audio_projector.state_dict(), out_path)
    print(f"Saved audio adapter: {out_path}")


def save_snac_heads(model, out_path):
    torch.save(model.snac_heads.state_dict(), out_path)
    print(f"Saved SNAC heads: {out_path}")

# --- Example ---
# save_audio_adapter(shruti_model, f"{DIRS['checkpoints']}/audio_adapter.pt")
# save_snac_heads(shruti_model, f"{DIRS['checkpoints']}/snac_heads.pt")
# reconstruct_and_save_audio(shruti_model, snac_codec, sample_features, sample_mask,
#                             f"{DIRS['checkpoints']}/sample_reconstruction.wav")

## Section 5 — Intent Vectorization & Zero-Persistence Storage Simulation (Milestone 6)

In [ ]:
import os, gc
import numpy as np
from cryptography.hazmat.primitives.ciphers.aead import AESGCM


def persist_ciphered_vector(vector_tensor: torch.Tensor, device_master_key: bytes) -> bytes:
    """Encrypts the non-invertible 512-d intent vector with AES-256-GCM and scrubs RAM."""
    raw_vector_bytes = vector_tensor.detach().cpu().numpy().astype("float32").tobytes()
    assert len(raw_vector_bytes) == 2048, f"Expected 2048 bytes, got {len(raw_vector_bytes)}"

    aesgcm = AESGCM(device_master_key)
    iv = os.urandom(12)
    ciphertext = aesgcm.encrypt(iv, raw_vector_bytes, None)
    payload = iv + ciphertext

    del raw_vector_bytes, vector_tensor
    gc.collect()
    return payload


def decrypt_vector(payload: bytes, device_master_key: bytes) -> np.ndarray:
    aesgcm = AESGCM(device_master_key)
    iv, ciphertext = payload[:12], payload[12:]
    raw = aesgcm.decrypt(iv, ciphertext, None)
    return np.frombuffer(raw, dtype=np.float32)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


def evaluate_retrieval_mrr(query_vectors: list, stored_vectors: list, ground_truth_idx: list) -> float:
    """Mean Reciprocal Rank across a set of debrief queries vs. stored intent vectors."""
    reciprocal_ranks = []
    for q_vec, gt_idx in zip(query_vectors, ground_truth_idx):
        scores = [cosine_similarity(q_vec, s_vec) for s_vec in stored_vectors]
        ranked_idx = np.argsort(scores)[::-1]
        rank = int(np.where(ranked_idx == gt_idx)[0][0]) + 1
        reciprocal_ranks.append(1.0 / rank)
    return float(np.mean(reciprocal_ranks))

# --- Example (requires trained model + a 256-bit device key) ---
# device_master_key = AESGCM.generate_key(bit_length=256)
# intent_vec = shruti_model.extract_intent_vector(hidden_states)
# cipher_blob = persist_ciphered_vector(intent_vec[0], device_master_key)
# recovered = decrypt_vector(cipher_blob, device_master_key)
# mrr = evaluate_retrieval_mrr(query_vectors, stored_vectors, ground_truth_idx)
# print(f"Retrieval MRR: {mrr:.3f} (target >= 0.82)")

## Section 6 — Multi-Part ONNX Export & INT4 Quantization (Milestone 7)

In [ ]:
import onnx
import onnxruntime as ort


# HF modules return ModelOutput dataclasses; torch.onnx.export needs a plain tensor/tuple.
class WhisperEncoderONNXWrapper(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, input_features):
        return self.encoder(input_features, return_dict=False)[0]


class QwenBackboneONNXWrapper(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, inputs_embeds, attention_mask):
        return self.backbone(inputs_embeds=inputs_embeds, attention_mask=attention_mask,
                              return_dict=False)[0]


def export_whisper_encoder_onnx(whisper_encoder, out_path, sample_len=3000, feature_dim=80):
    wrapped = WhisperEncoderONNXWrapper(whisper_encoder).to(DEVICE).eval()
    dummy_input = torch.randn(1, feature_dim, sample_len, device=DEVICE)
    torch.onnx.export(
        wrapped, dummy_input, out_path,
        input_names=["input_features"], output_names=["encoder_hidden_states"],
        dynamic_axes={"input_features": {2: "seq_len"}, "encoder_hidden_states": {1: "seq_len"}},
        opset_version=18,
    )
    onnx.checker.check_model(onnx.load(out_path))
    print(f"Exported + validated: {out_path}")


def export_snac_decoder_onnx(snac_codec, out_path, num_tokens=100, num_codebooks=7, codebook_size=4096):
    dummy_codes = [torch.randint(0, codebook_size, (1, num_tokens), device=DEVICE)
                   for _ in range(num_codebooks)]
    torch.onnx.export(
        snac_codec.decoder, tuple(dummy_codes), out_path,
        input_names=[f"codes_{i}" for i in range(num_codebooks)], output_names=["audio"],
        opset_version=18,
    )
    onnx.checker.check_model(onnx.load(out_path))
    print(f"Exported + validated: {out_path}")


def export_qwen_backbone_onnx(shruti_model, out_path, hidden_size=2048, seq_len=64):
    wrapped = QwenBackboneONNXWrapper(shruti_model.backbone).to(DEVICE).eval()
    dummy_embeds = torch.randn(1, seq_len, hidden_size, device=DEVICE)
    dummy_mask = torch.ones(1, seq_len, dtype=torch.long, device=DEVICE)
    torch.onnx.export(
        wrapped, (dummy_embeds, dummy_mask), out_path,
        input_names=["inputs_embeds", "attention_mask"], output_names=["hidden_states"],
        dynamic_axes={"inputs_embeds": {1: "seq_len"}, "attention_mask": {1: "seq_len"},
                       "hidden_states": {1: "seq_len"}},
        opset_version=18,
    )
    onnx.checker.check_model(onnx.load(out_path))
    print(f"Exported + validated: {out_path}")
    print("Quantize with ONNX Runtime GenAI model builder for INT4 AWQ/block quantization:")
    print("  python -m onnxruntime_genai.models.builder -i <hf_ckpt> -o <out_dir> "
          "-p int4 -e cpu")


def validate_onnx_vs_torch(torch_model, onnx_path, dummy_inputs: tuple, atol=1e-3):
    torch_model.eval()
    with torch.no_grad():
        torch_out = torch_model(*dummy_inputs)
        torch_out = torch_out[0] if isinstance(torch_out, tuple) else torch_out
    session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    input_names = [inp.name for inp in session.get_inputs()]
    ort_inputs = {name: t.cpu().numpy() for name, t in zip(input_names, dummy_inputs)}
    ort_out = session.run(None, ort_inputs)[0]
    mse = float(np.mean((torch_out.cpu().numpy() - ort_out) ** 2))
    print(f"PyTorch vs ONNX MSE: {mse:.6f} (target <= 1e-3)")
    return mse

# --- Example ---
# export_whisper_encoder_onnx(whisper_encoder, f"{DIRS['onnx']}/whisper_encoder.onnx")
# export_snac_decoder_onnx(snac_codec, f"{DIRS['onnx']}/snac_decoder.onnx")
# export_qwen_backbone_onnx(shruti_model, f"{DIRS['onnx']}/qwen3_omni_backbone.onnx")

In [ ]:
def check_onnx_package_size(onnx_dir: str, target_mb: float = 780.0):
    """Sums exported ONNX artifact sizes against the Milestone 7 mobile package budget."""
    total_bytes = 0
    for fname in ("whisper_encoder.onnx", "qwen3_omni_backbone.onnx", "snac_decoder.onnx"):
        fpath = os.path.join(onnx_dir, fname)
        if os.path.exists(fpath):
            fsize = os.path.getsize(fpath)
            print(f"  {fname}: {fsize / 1e6:.1f} MB")
            total_bytes += fsize
        else:
            print(f"  {fname}: not found")
    total_mb = total_bytes / 1e6
    status = "OK" if total_mb <= target_mb else "OVER BUDGET"
    print(f"Total ONNX package size: {total_mb:.1f} MB (target <= {target_mb} MB) [{status}]")
    return total_mb

# --- Example ---
# check_onnx_package_size(DIRS["onnx"])

## Section 7 — Speaker Preset Export & Mobile Integration Handoff (Milestone 8)

The remaining Milestone 8 work — the JNI bridge, C++ barge-in fast-path (VAD + AEC + ring
buffer), KV-cache rollback, and SQLCipher/AndroidKeystore wiring — lives in the Android
app's Kotlin/C++ codebase, not in this training notebook (see `milestone_tasks.md`,
Milestone 8). This section covers only the one Python-side deliverable: extracting the
512-d speaker conditioning vectors bundled as `.bin` assets for the SNAC decoder.

In [ ]:
def extract_speaker_embedding(reference_wav_path: str, whisper_encoder, whisper_fe,
                               intent_proj: nn.Linear) -> np.ndarray:
    """Derives a 512-d speaker/style conditioning vector from a ~5s reference clip,
    reusing the intent projection head as a generic pooling+projection layer."""
    import soundfile as sf
    audio, sr = sf.read(reference_wav_path)
    inputs = whisper_fe(audio, sampling_rate=sr, return_tensors="pt").input_features.to(DEVICE)
    with torch.no_grad():
        features = whisper_encoder(inputs, return_dict=False)[0]
        pooled = features.mean(dim=1)
        vector = intent_proj(pooled)
        vector = vector / vector.norm(p=2, dim=-1, keepdim=True)
    return vector.squeeze(0).cpu().numpy().astype("float32")


def save_speaker_preset(vector: np.ndarray, out_path: str):
    assert vector.shape == (512,), f"Expected 512-d vector, got {vector.shape}"
    with open(out_path, "wb") as f:
        f.write(vector.tobytes())
    print(f"Saved speaker preset: {out_path} ({os.path.getsize(out_path)} bytes)")

# --- Example ---
# for name, wav in [("aditi", "aditi_ref.wav"), ("agastya", "agastya_ref.wav"),
#                    ("priya", "priya_ref.wav"), ("kabir", "kabir_ref.wav")]:
#     vec = extract_speaker_embedding(wav, whisper_encoder, whisper_fe, shruti_model.intent_proj)
#     save_speaker_preset(vec, f"{DIRS['voices']}/{name}.bin")

## Section 8 — Validation & Test Suite

Fast, self-contained `unittest` checks for the pure-Python/NumPy utilities, audio DSP
functions, and the `ShrutiS2SModel` wiring — all run against synthetic data so they work
without GPU access, network calls, or downloading Qwen/Whisper/SNAC weights. Heavier
pipeline checks (model pruning, ONNX export parity, live TTS calls) are included as
skipped integration-test stubs; flip `RUN_INTEGRATION_TESTS = True` once a GPU runtime
with the full dependency stack and model downloads is available.

In [ ]:
import unittest
import tempfile


class TestUtilityFunctions(unittest.TestCase):
    def test_pad_or_truncate_pads(self):
        result = _pad_or_truncate(np.array([1, 2, 3]), 5)
        np.testing.assert_array_equal(result, [1, 2, 3, 0, 0])

    def test_pad_or_truncate_truncates(self):
        result = _pad_or_truncate(np.array([1, 2, 3, 4, 5]), 3)
        np.testing.assert_array_equal(result, [1, 2, 3])

    def test_compute_wer_identical(self):
        self.assertEqual(compute_wer("hello there", "hello there"), 0.0)

    def test_compute_wer_full_mismatch(self):
        self.assertAlmostEqual(compute_wer("a b c", "x y z"), 1.0)

    def test_compute_wer_partial_mismatch(self):
        self.assertAlmostEqual(compute_wer("the quick brown fox", "the quick red fox"), 1 / 4)

    def test_cosine_similarity_identical_vectors(self):
        v = np.array([1.0, 2.0, 3.0])
        self.assertAlmostEqual(cosine_similarity(v, v), 1.0, places=5)

    def test_cosine_similarity_orthogonal_vectors(self):
        sim = cosine_similarity(np.array([1.0, 0.0]), np.array([0.0, 1.0]))
        self.assertAlmostEqual(sim, 0.0, places=5)

    def test_evaluate_retrieval_mrr_perfect_rank(self):
        stored = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([-1.0, 0.0])]
        mrr = evaluate_retrieval_mrr([np.array([1.0, 0.0])], stored, ground_truth_idx=[0])
        self.assertAlmostEqual(mrr, 1.0, places=5)

    def test_telephony_bandpass_preserves_length(self):
        sr = 16000
        audio = np.random.randn(sr).astype(np.float32)
        filtered = telephony_bandpass(audio, sr)
        self.assertEqual(filtered.shape, audio.shape)

    def test_mix_background_noise_preserves_length(self):
        audio = np.ones(1000, dtype=np.float32)
        noise = np.random.randn(200).astype(np.float32)
        mixed = mix_background_noise(audio, noise, snr_db=10)
        self.assertEqual(mixed.shape, audio.shape)

    def test_aes_gcm_round_trip(self):
        from cryptography.hazmat.primitives.ciphers.aead import AESGCM
        key = AESGCM.generate_key(bit_length=256)
        vector = torch.nn.functional.normalize(torch.randn(512), dim=0)
        payload = persist_ciphered_vector(vector.clone(), key)
        recovered = decrypt_vector(payload, key)
        np.testing.assert_allclose(recovered, vector.numpy(), atol=1e-6)

    def test_check_onnx_package_size_under_budget(self):
        with tempfile.TemporaryDirectory() as tmp_dir:
            for fname, size_mb in [("whisper_encoder.onnx", 50), ("qwen3_omni_backbone.onnx", 400),
                                    ("snac_decoder.onnx", 30)]:
                with open(os.path.join(tmp_dir, fname), "wb") as f:
                    f.write(b"0" * int(size_mb * 1e6))
            total_mb = check_onnx_package_size(tmp_dir, target_mb=780.0)
            self.assertLess(total_mb, 780.0)


class TestAudioPipeline(unittest.TestCase):
    def _write_tone(self, path, freq=440, sr=16000, duration=0.5):
        import soundfile as sf
        t = np.linspace(0, duration, int(sr * duration), endpoint=False)
        tone = (0.1 * np.sin(2 * np.pi * freq * t)).astype(np.float32)
        sf.write(path, tone, sr)
        return tone

    def test_augment_telephony_audio_preserves_duration(self):
        import soundfile as sf
        with tempfile.TemporaryDirectory() as tmp_dir:
            in_path, out_path = f"{tmp_dir}/in.wav", f"{tmp_dir}/out.wav"
            tone = self._write_tone(in_path)
            augment_telephony_audio(in_path, out_path)
            augmented, sr = sf.read(out_path)
            self.assertEqual(sr, 16000)
            self.assertEqual(len(augmented), len(tone))

    def test_splice_interruption_extends_audio_at_cut_point(self):
        import soundfile as sf
        with tempfile.TemporaryDirectory() as tmp_dir:
            assistant_path, caller_path = f"{tmp_dir}/assistant.wav", f"{tmp_dir}/caller.wav"
            out_path = f"{tmp_dir}/spliced.wav"
            self._write_tone(assistant_path, duration=3.0)
            caller_tone = self._write_tone(caller_path, freq=880, duration=1.0)
            cut_seconds = splice_interruption(assistant_path, caller_path, out_path, cut_range=(1.0, 1.0))
            spliced, sr = sf.read(out_path)
            self.assertAlmostEqual(cut_seconds, 1.0, places=2)
            self.assertEqual(len(spliced), int(1.0 * sr) + len(caller_tone))

    def test_pack_dataset_to_parquet_round_trip(self):
        import pandas as pd
        with tempfile.TemporaryDirectory() as tmp_dir:
            out_path = f"{tmp_dir}/dataset.parquet"
            records = [{"sample_id": 0, "scenario": "delivery_courier", "language": "en-IN",
                        "turn_paths": ["a.wav"], "has_interruption": False, "cut_seconds": None}]
            pack_dataset_to_parquet(records, out_path)
            df = pd.read_parquet(out_path)
            self.assertEqual(len(df), 1)
            self.assertEqual(df.iloc[0]["scenario"], "delivery_courier")

In [ ]:
class _DummyBackboneOutput:
    def __init__(self, last_hidden_state):
        self.last_hidden_state = last_hidden_state


class _DummyQwenModel(nn.Module):
    """Stand-in for the pruned Qwen backbone so architecture tests don't need model downloads."""

    def __init__(self, hidden_size):
        super().__init__()
        self.proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, inputs_embeds, attention_mask=None):
        return _DummyBackboneOutput(self.proj(inputs_embeds))


class _DummyPrunedBackbone:
    def __init__(self, hidden_size=64):
        self.config = type("DummyConfig", (), {"hidden_size": hidden_size})()
        self.model = _DummyQwenModel(hidden_size)


class TestShrutiS2SModelArchitecture(unittest.TestCase):
    def setUp(self):
        self.hidden_size = 64
        self.model = ShrutiS2SModel(
            _DummyPrunedBackbone(self.hidden_size),
            whisper_dim=32, snac_codebook_size=16, num_snac_heads=3, intent_dim=8,
        )

    def test_forward_audio_features_shape(self):
        features = torch.randn(2, 10, 32)
        mask = torch.ones(2, 10, dtype=torch.long)
        hidden = self.model.forward_audio_features(features, mask)
        self.assertEqual(hidden.shape, (2, 10, self.hidden_size))

    def test_predict_snac_tokens_shapes(self):
        hidden = self.model.forward_audio_features(torch.randn(2, 10, 32))
        logits_per_codebook = self.model.predict_snac_tokens(hidden)
        self.assertEqual(len(logits_per_codebook), 3)
        for logits in logits_per_codebook:
            self.assertEqual(logits.shape, (2, 10, 16))

    def test_extract_intent_vector_is_unit_norm(self):
        hidden = self.model.forward_audio_features(torch.randn(3, 5, 32))
        vector = self.model.extract_intent_vector(hidden)
        self.assertEqual(vector.shape, (3, 8))
        torch.testing.assert_close(vector.norm(p=2, dim=-1), torch.ones(3), atol=1e-5, rtol=0)

    def test_extract_intent_vector_respects_attention_mask(self):
        features = torch.randn(1, 6, 32)
        mask = torch.tensor([[1, 1, 1, 0, 0, 0]])
        hidden = self.model.forward_audio_features(features, mask)
        vector_masked = self.model.extract_intent_vector(hidden, mask)
        vector_unmasked = self.model.extract_intent_vector(hidden)
        self.assertFalse(torch.allclose(vector_masked, vector_unmasked))

In [ ]:
RUN_INTEGRATION_TESTS = False  # requires GPU + network + Qwen/Whisper/SNAC downloads


@unittest.skipUnless(RUN_INTEGRATION_TESTS, "requires model downloads + GPU/network")
class TestIntegrationPipeline(unittest.TestCase):
    def test_prune_qwen_layer_and_vocab_counts(self):
        pruned = prune_qwen_for_shruti(target_layers=22, num_special_tokens=512)
        self.assertEqual(pruned.config.num_hidden_layers, 22)
        self.assertEqual(pruned.config.vocab_size, 512)
        self.assertIsNone(pruned.lm_head)

    def test_frozen_whisper_and_snac_load(self):
        whisper_encoder, _ = load_frozen_whisper_encoder()
        snac_codec = load_snac_codec()
        self.assertFalse(any(p.requires_grad for p in whisper_encoder.parameters()))
        self.assertFalse(any(p.requires_grad for p in snac_codec.parameters()))

    def test_onnx_export_matches_torch_within_tolerance(self):
        pruned = prune_qwen_for_shruti()
        shruti_model = ShrutiS2SModel(pruned).to(DEVICE)
        out_path = f"{DIRS['onnx']}/qwen3_omni_backbone_test.onnx"
        export_qwen_backbone_onnx(shruti_model, out_path)
        wrapped = QwenBackboneONNXWrapper(shruti_model.backbone).to(DEVICE).eval()
        dummy_embeds = torch.randn(1, 64, shruti_model.backbone.config.hidden_size, device=DEVICE)
        dummy_mask = torch.ones(1, 64, dtype=torch.long, device=DEVICE)
        mse = validate_onnx_vs_torch(wrapped, out_path, (dummy_embeds, dummy_mask))
        self.assertLessEqual(mse, 1e-3)

In [ ]:
def run_validation_suite(verbosity: int = 2) -> unittest.TestResult:
    """Runs the offline unit tests plus any enabled integration tests."""
    loader = unittest.TestLoader()
    suite = unittest.TestSuite([
        loader.loadTestsFromTestCase(TestUtilityFunctions),
        loader.loadTestsFromTestCase(TestAudioPipeline),
        loader.loadTestsFromTestCase(TestShrutiS2SModelArchitecture),
        loader.loadTestsFromTestCase(TestIntegrationPipeline),
    ])
    result = unittest.TextTestRunner(verbosity=verbosity).run(suite)
    status = "PASSED" if result.wasSuccessful() else "FAILED"
    print(f"\nValidation suite {status}: {result.testsRun} tests, "
          f"{len(result.failures)} failures, {len(result.errors)} errors")
    return result


run_validation_suite()